In [1]:
import pickle
import os
import pandas as pd
import sys

In [2]:
prop_res = pickle.load(open('../Matskraft_property/data/res_dir_without_threshold/matskraft_property.pkl', 'rb'))
gold_tuples = pickle.load(open('../Matskraft_property/code/test_gold_tuples.pkl', 'rb'))

In [3]:
print(prop_res.keys())
print(prop_res['infer'].keys())
print(len(prop_res['infer']['pred_tuples']))
print(len(gold_tuples))

dict_keys(['infer'])
dict_keys(['identifier', 'y_comp_pred', 'ret_comp_pred', 'ret_tuples_pred', 'pred_tuples', 'gold_tuples'])
3389
3520


In [4]:
train_test_split = pickle.load(open('../Matskraft_property/data/train_val_test_split_for_prop.pkl', 'rb'))
make_pii_test = []
for (pii, tidx) in train_test_split['test']:
    make_pii_test.append(pii+'_'+str(tidx))

In [5]:
prop_pred_tuples_test, prop_gold_tuples_test = [], gold_tuples

for tup in prop_res['infer']['pred_tuples']:
    key = "_".join(tup[0].split("_")[:2])
    if key in make_pii_test:
        prop_pred_tuples_test.append(tup)
        
print(len(prop_pred_tuples_test), len(prop_gold_tuples_test))

3389 3520


In [6]:
prop_pred_tuples_test_set = set(prop_pred_tuples_test)
print(len(prop_pred_tuples_test_set))

3389


In [7]:
def match_tuple(p, g, split):
    
    if p[0].count('_') == 4:
        p0 = p[0][:p[0].rindex('_')]
    else:
        p0 = p[0]
        
    if g[0].count('_') == 4:
        g0 = g[0][:g[0].rindex('_')]
    else:
        g0 = g[0]
        
#     p0 = p[0][:p[0].rindex('_')]
#     g0 = g[0][:g[0].rindex('_')]
#     print(f'{p[0]}  ---->  {p0}')
#     print(f'{g[0]}  ---->  {g0}')
#     print()
#     if p0 == g0 and p[0]!=g[0]:
#         print(p)
#         print(g)
#         print()
    try:
        return p0 == g0 and p[1] == g[1] and round(abs(p[2] - g[2]),2)==0 and p[3] == g[3]
    except TypeError:
        print('Check for type mismatch')
        print(p)
        print(g)
        return False

# def get_tuples_metrics(gold_tuples, pred_tuples, split):
#     global mismatch_train, mismatch_val, mismatch_test
#     true_positives, false_positives = [], []
#     prec = 0
#     for p in pred_tuples:
#         for g in gold_tuples:
#             if match_tuple(p, g, split):
#                 prec += 1
#                 break
#         # if p in gold_tuples:
#         #     prec += 1
#     if len(pred_tuples) > 0:
#         prec /= len(pred_tuples)
#     else:
#         prec = 0.0
#     rec = 0
#     for g in gold_tuples:
#         for p in pred_tuples:
#             if match_tuple(p, g, split):
#                 rec += 1
#                 break
#         # if g in pred_tuples:
#         #     rec += 1
# #     new_gold_tup = [tup for tup in gold_tuples if tup[-2]!='']
#     rec /= len(gold_tuples)
#     fscore = 2 * prec * rec / (prec + rec) if (prec + rec > 0) else 0.0

#     return {'precision': round(prec,4), 'recall': round(rec,4), 'fscore': round(fscore,4)}

def get_tuples_metrics(gold_tuples, pred_tuples, split):
    global mismatch_train, mismatch_val, mismatch_test
    true_positives, false_positives, false_negatives = [], [], []
    
    # Calculate precision and collect true/false positives
    prec = 0
    for p in pred_tuples:
        found_match = False
        for g in gold_tuples:
            if match_tuple(p, g, split):
                prec += 1
                true_positives.append((p))  # Store the matched pair
                found_match = True
                break
        if not found_match:
            false_positives.append(p)  # Store unmatched prediction
    
    if len(pred_tuples) > 0:
        prec /= len(pred_tuples)
    else:
        prec = 0.0
    
    # Calculate recall and collect false negatives
    rec = 0
    for g in gold_tuples:
        found_match = False
        for p in pred_tuples:
            if match_tuple(p, g, split):
                rec += 1
                found_match = True
                break
        if not found_match:
            false_negatives.append(g)  # Store unmatched gold standard
    
    rec /= len(gold_tuples)
    fscore = 2 * prec * rec / (prec + rec) if (prec + rec > 0) else 0.0
    
    return {
        'precision': round(prec, 4), 
        'recall': round(rec, 4), 
        'fscore': round(fscore, 4),
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives
    }

In [8]:
import pickle  # Add this at the top of your file if not already present

split = ''
tuple_metrics = get_tuples_metrics(prop_gold_tuples_test, prop_pred_tuples_test, split)
# Print only the metrics, not the entire dictionary
print(f"Precision: {tuple_metrics['precision']}")
print(f"Recall: {tuple_metrics['recall']}")
print(f"F1 Score: {tuple_metrics['fscore']}")

Precision: 0.9035
Recall: 0.8707
F1 Score: 0.8868


In [10]:
pickle.dump(prop_gold_tuples_test, open('prop_gold_tuples_test.pkl', 'wb'))